# Exercise 5: copy/compute overlap and tail latency

## Question

Can a GPU beamforming pipeline sustain a stated input rate, and what happens to frame latency, queueing, and drops as the input rate approaches or exceeds service capacity?

This notebook replays measured PyMUST rotating-disk phantom data. It is a controlled replay experiment, not a live scanner feed. The RF-to-IQ conversion and geometry setup happen once before timing. Each replay frame is one receive acquisition processed by Mach; the GPU accumulates its power contribution and copies the final map to the host once per trial.


## What the two paths measure

| Path | Per-frame data movement | Scheduling | Purpose |
|---|---|---|---|
| Serialized | pinned host buffer → device frame buffer | wait for each copy, beamform, and power update to finish before accepting the next frame | baseline with transfer and compute serialized |
| Overlapped | pinned host buffer → one of two device frame buffers | copy stream feeds a compute stream; events protect each buffer while the GPU is using it | test whether the copy of a later frame can overlap beamforming of the current frame |

A producer schedules frame arrivals at a configured rate. A bounded queue holds at most eight waiting frames; when full, the newest arrival is dropped. The overlap path has two frames in flight at most. The rate sweep spans below, near, and above the recording's nominal 10 kHz PRF. The source contains 32 acquisitions, replayed for 64 cycles per trial.

The report separates requested rate from actual producer timing. It records producer lateness, queue wait, host-observed frame latency, H2D and GPU event durations, deadline misses, queue depth, and drops. The deadline is one requested input period after each scheduled arrival. GPU event timings exclude producer and host scheduling time.

These are replayed stored frames with repeated content; they do not represent live acquisition timing or independent biological data. Python scheduling jitter, Mach's per-frame launch overhead, and the small single-acquisition payload are part of this experiment. Use Nsight Systems separately to check that the CUDA copy and compute lanes overlap before describing a measured benefit as actual GPU overlap.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CYCLES_PER_TRIAL = 64
INPUT_RATES_HZ = (1_000, 3_000, 6_000, 10_000, 15_000)
QUEUE_CAPACITY = 8
REPLAYED_PRF_HZ = 10_000
POLL_INTERVAL_S = 0.0001
GRID_SIZE = 251

_candidates = [Path("/content/realtime-beamforming-lab"), Path.cwd(), *Path.cwd().resolve().parents]
ROOT = next((p.resolve() for p in _candidates if (p / "pyproject.toml").is_file()), Path.cwd().resolve())
RESULTS_DIR = ROOT / "results"
ARTIFACTS_DIR = ROOT / "artifacts"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

print("Repository:", ROOT)
print("Cycles per trial:", CYCLES_PER_TRIAL)
print("Offered rates (Hz):", INPUT_RATES_HZ)
print("Queue capacity:", QUEUE_CAPACITY)


### Install the Colab dependencies

Run this on the connected Colab T4 runtime. If pip requests a kernel restart, restart before continuing. The local lab source is imported from `src/` in the setup cell above.


In [ ]:
%pip uninstall -y cupy-cuda12x cupy-cuda13x
%pip install --no-cache-dir "mach-beamform[examples,uff,vis]>=0.2.1,<0.3.0" "cupy-cuda13x>=13.0" "pymust>=0.1.9" "psutil>=5.9" "matplotlib>=3.9"


In [ ]:
import importlib.metadata as package_metadata
import json
import platform
import time

import cupy as cp
import numpy as np
import psutil

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("No CUDA device found. Connect this notebook to the Colab T4 runtime.")

_device_name = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)["name"]
if isinstance(_device_name, bytes):
    _device_name = _device_name.decode(errors="replace")
_git = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True, capture_output=True)
_versions = {}
for _name in ("mach-beamform", "cupy-cuda13x", "PyMUST", "numpy"):
    try:
        _versions[_name] = package_metadata.version(_name)
    except package_metadata.PackageNotFoundError:
        _versions[_name] = "not installed"

runtime_metadata = {
    "device": str(_device_name),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cpu_logical_count": os.cpu_count(),
    "memory_total_bytes": psutil.virtual_memory().total,
    "git_revision": _git.stdout.strip() if _git.returncode == 0 else None,
    "versions": _versions,
}
print(json.dumps(runtime_metadata, indent=2))
subprocess.run(["nvidia-smi"], check=False)


## Load the measured replay source

The source helper downloads/caches the PyMUST rotating-disk phantom, converts its RF to complex IQ once, and prepares the scan geometry. Those steps are outside all streaming timings. The per-frame transfer uses complex64 IQ, matching the existing Mach workload. The nominal source PRF is 10 kHz; the recorded file itself does not contain live timestamps, so arrivals below are generated by this notebook.


In [ ]:
from beamforming_lab.beamformer import prepare_geometry, run_ensemble
from beamforming_lab.workload import load_pymust_doppler_workload

workload = load_pymust_doppler_workload(grid_size=GRID_SIZE)
geometry = prepare_geometry(workload, "cupy")
channel_data = workload.iq_elements_samples_frames
frame_count = int(channel_data.shape[2])
frame_shape = (int(channel_data.shape[0]), int(channel_data.shape[1]), 1)
frame_bytes = int(np.prod(frame_shape) * np.dtype(np.complex64).itemsize)
replay_frames = frame_count * CYCLES_PER_TRIAL

print("Dataset:", workload.data_provenance["dataset_id"])
print("Recording note:", workload.data_provenance["recording"])
print("IQ shape (elements, samples, frames):", channel_data.shape)
print("IQ dtype:", channel_data.dtype)
print(f"One replay frame: {frame_bytes:,} bytes ({frame_bytes / 2**20:.3f} MiB)")
print("Frames per trial:", replay_frames)
print(f"Nominal {REPLAYED_PRF_HZ:,} Hz period: {1e3 / REPLAYED_PRF_HZ:.3f} ms")
print(f"RF-to-IQ setup time (excluded): {workload.rf_to_iq_ms:.2f} ms")
print(f"Geometry upload time (excluded): {geometry.geometry_upload_ms:.2f} ms")


## Replay implementation

The source thread is open-loop: it schedules arrivals against a monotonic clock and uses a hybrid sleep/spin wait for sub-millisecond periods. Actual arrival jitter is saved alongside requested rate. The consumer applies a drop-newest policy to the bounded queue.

The serialized path waits for each H2D event before submitting compute, then waits for each frame's GPU work to complete. The overlap path queues H2D on a copy stream and makes the compute stream wait on a CUDA event; two pinned host and device slots prevent a frame buffer from being overwritten while in use. Mach beamforming plus the GPU-side power reduction are included in each compute event. Geometry stays resident, and no frame-level D2H occurs. The stream and event calls follow CuPy's [stream](https://docs.cupy.dev/en/stable/reference/generated/cupy.cuda.Stream.html) and [`memcpyAsync`](https://docs.cupy.dev/en/stable/reference/generated/cupy.cuda.runtime.memcpyAsync.html) APIs.


In [ ]:
from dataclasses import dataclass
from queue import Empty, Full, Queue
from threading import Event, Thread

from mach.kernel import beamform

try:
    from cupy.cuda.nvtx import RangePop, RangePush
    NVTX_AVAILABLE = True
except ImportError:
    RangePop = RangePush = None
    NVTX_AVAILABLE = False

@dataclass
class StreamSlot:
    owner: object
    host: np.ndarray
    device: object
    copy_stream: object
    compute_stream: object
    copy_start: object
    copy_done: object
    compute_start: object
    compute_done: object
    job: dict | None = None
    host_stage_ms: float = 0.0


def make_slot():
    owner = cp.cuda.alloc_pinned_memory(frame_bytes)
    host = np.frombuffer(owner, dtype=np.complex64).reshape(frame_shape)
    device = cp.empty(frame_shape, dtype=cp.complex64)
    copy_stream = cp.cuda.Stream(non_blocking=True)
    compute_stream = cp.cuda.Stream(non_blocking=True)
    return StreamSlot(
        owner=owner,
        host=host,
        device=device,
        copy_stream=copy_stream,
        compute_stream=compute_stream,
        copy_start=cp.cuda.Event(),
        copy_done=cp.cuda.Event(),
        compute_start=cp.cuda.Event(),
        compute_done=cp.cuda.Event(),
    )


def _gpu_warmup():
    warm_slot = make_slot()
    np.copyto(warm_slot.host, channel_data[:, :, :1])
    with warm_slot.copy_stream:
        cp.cuda.runtime.memcpyAsync(
            int(warm_slot.device.data.ptr),
            int(warm_slot.host.ctypes.data),
            frame_bytes,
            cp.cuda.runtime.memcpyHostToDevice,
            int(warm_slot.copy_stream.ptr),
        )
    warm_slot.copy_stream.synchronize()
    with warm_slot.compute_stream:
        image = beamform(
            channel_data=warm_slot.device,
            rx_coords_m=geometry.rx_coords_m,
            scan_coords_m=geometry.scan_coords_m,
            tx_wave_arrivals_s=geometry.tx_wave_arrivals_s,
            f_number=float(workload.params["fnumber"]),
            rx_start_s=float(workload.params["t0"]),
            sampling_freq_hz=float(workload.params["fs"]),
            sound_speed_m_s=float(workload.params["c"]),
            modulation_freq_hz=float(workload.params["fc"]),
            tukey_alpha=0.0,
        )
        cp.sum(cp.abs(image) ** 2, axis=-1)
    warm_slot.compute_stream.synchronize()
    del warm_slot


def run_trial(mode: str, input_rate_hz: int) -> dict:
    if mode not in {"serialized", "overlapped"}:
        raise ValueError("mode must be 'serialized' or 'overlapped'")
    if input_rate_hz <= 0:
        raise ValueError("input_rate_hz must be positive")

    slot_count = 1 if mode == "serialized" else 2
    slots = [make_slot() for _ in range(slot_count)]
    power_accumulator = cp.zeros(len(workload.scan_coords_m), dtype=cp.float32)
    cp.cuda.Device().synchronize()
    work_queue = Queue(maxsize=QUEUE_CAPACITY)
    producer_done = Event()
    arrivals = []
    drops = []
    completed = []
    max_queue_depth = 0
    period_ns = int(round(1e9 / input_rate_hz))
    offered_count = replay_frames
    trial_start_ns = time.perf_counter_ns()

    def produce():
        nonlocal max_queue_depth
        try:
            for sequence in range(offered_count):
                scheduled_ns = trial_start_ns + sequence * period_ns
                remaining_ns = scheduled_ns - time.perf_counter_ns()
                if remaining_ns > 200_000:
                    time.sleep((remaining_ns - 200_000) / 1e9)
                while time.perf_counter_ns() < scheduled_ns:
                    pass
                arrived_ns = time.perf_counter_ns()
                job = {
                    "sequence": sequence,
                    "frame_index": sequence % frame_count,
                    "scheduled_ns": scheduled_ns,
                    "arrived_ns": arrived_ns,
                    "queue_depth_before_enqueue": work_queue.qsize(),
                }
                arrivals.append(job)
                try:
                    work_queue.put_nowait(job)
                    max_queue_depth = max(max_queue_depth, work_queue.qsize())
                except Full:
                    drops.append({**job, "queue_depth_at_drop": work_queue.qsize(), "reason": "drop_newest_queue_full"})
        finally:
            producer_done.set()

    def submit(slot: StreamSlot, job: dict):
        stage_start_ns = time.perf_counter_ns()
        np.copyto(slot.host, channel_data[:, :, job["frame_index"]:job["frame_index"] + 1])
        slot.host_stage_ms = (time.perf_counter_ns() - stage_start_ns) / 1e6
        slot.job = {**job, "stage_start_ns": stage_start_ns}

        with slot.copy_stream:
            slot.copy_start.record(slot.copy_stream)
            cp.cuda.runtime.memcpyAsync(
                int(slot.device.data.ptr),
                int(slot.host.ctypes.data),
                frame_bytes,
                cp.cuda.runtime.memcpyHostToDevice,
                int(slot.copy_stream.ptr),
            )
            slot.copy_done.record(slot.copy_stream)

        if mode == "serialized":
            slot.copy_done.synchronize()
        else:
            slot.compute_stream.wait_event(slot.copy_done)

        with slot.compute_stream:
            slot.compute_start.record(slot.compute_stream)
            image = beamform(
                channel_data=slot.device,
                rx_coords_m=geometry.rx_coords_m,
                scan_coords_m=geometry.scan_coords_m,
                tx_wave_arrivals_s=geometry.tx_wave_arrivals_s,
                f_number=float(workload.params["fnumber"]),
                rx_start_s=float(workload.params["t0"]),
                sampling_freq_hz=float(workload.params["fs"]),
                sound_speed_m_s=float(workload.params["c"]),
                modulation_freq_hz=float(workload.params["fc"]),
                tukey_alpha=0.0,
            )
            frame_power = cp.sum(cp.abs(image) ** 2, axis=-1)
            cp.add(power_accumulator, frame_power, out=power_accumulator)
            slot.compute_done.record(slot.compute_stream)

        if mode == "serialized":
            slot.compute_done.synchronize()
            finish(slot)

    def finish(slot: StreamSlot):
        completed_ns = time.perf_counter_ns()
        job = slot.job
        completed.append({
            **job,
            "completed_ns": completed_ns,
            "queue_wait_ms": (job["stage_start_ns"] - job["arrived_ns"]) / 1e6,
            "host_latency_ms": (completed_ns - job["arrived_ns"]) / 1e6,
            "producer_lateness_ms": (job["arrived_ns"] - job["scheduled_ns"]) / 1e6,
            "h2d_ms": float(cp.cuda.get_elapsed_time(slot.copy_start, slot.copy_done)),
            "gpu_beamform_power_ms": float(cp.cuda.get_elapsed_time(slot.compute_start, slot.compute_done)),
            "host_stage_copy_ms": slot.host_stage_ms,
            "deadline_miss": completed_ns > job["scheduled_ns"] + period_ns,
        })
        slot.job = None

    _nvtx_pushed = False
    if NVTX_AVAILABLE:
        RangePush(f"exercise5/{mode}/{input_rate_hz}Hz")
        _nvtx_pushed = True
    producer = Thread(target=produce, name="exercise5-replay-producer", daemon=True)
    producer.start()
    try:
        while True:
            progressed = False
            for slot in slots:
                if slot.job is not None and slot.compute_done.query():
                    finish(slot)
                    progressed = True
            free_slot = next((slot for slot in slots if slot.job is None), None)
            if free_slot is not None:
                try:
                    job = work_queue.get_nowait()
                except Empty:
                    job = None
                if job is not None:
                    job["queue_depth_after_dequeue"] = work_queue.qsize()
                    submit(free_slot, job)
                    progressed = True
            max_queue_depth = max(max_queue_depth, work_queue.qsize())
            if producer_done.is_set() and work_queue.empty() and all(slot.job is None for slot in slots):
                break
            if not progressed:
                time.sleep(POLL_INTERVAL_S)
    finally:
        producer.join()
        for slot in slots:
            if slot.job is not None:
                slot.compute_done.synchronize()
                finish(slot)
        if _nvtx_pushed:
            RangePop()

    cp.cuda.Device().synchronize()
    power_map = cp.asnumpy(power_accumulator).reshape(len(workload.x_m), len(workload.z_m))
    trial_end_ns = time.perf_counter_ns()
    latencies = np.asarray([record["host_latency_ms"] for record in completed], dtype=np.float64)
    producer_lateness = np.asarray([record["producer_lateness_ms"] for record in arrivals], dtype=np.float64)
    h2d_times = np.asarray([record["h2d_ms"] for record in completed], dtype=np.float64)
    gpu_times = np.asarray([record["gpu_beamform_power_ms"] for record in completed], dtype=np.float64)
    accepted = len(completed)
    first_arrival_ns = arrivals[0]["arrived_ns"] if arrivals else trial_start_ns
    last_arrival_ns = arrivals[-1]["arrived_ns"] if arrivals else trial_start_ns
    actual_producer_rate = ((len(arrivals) - 1) * 1e9 / (last_arrival_ns - first_arrival_ns)
                            if len(arrivals) > 1 and last_arrival_ns > first_arrival_ns else 0.0)

    return {
        "mode": mode,
        "requested_rate_hz": input_rate_hz,
        "actual_producer_rate_hz": float(actual_producer_rate),
        "offered_frames": offered_count,
        "accepted_frames": accepted,
        "dropped_frames": len(drops),
        "deadline_misses": int(sum(record["deadline_miss"] for record in completed)),
        "deadline_miss_fraction": float(np.mean([record["deadline_miss"] for record in completed])) if accepted else None,
        "max_waiting_queue_depth": int(max_queue_depth),
        "trial_wall_s": (trial_end_ns - trial_start_ns) / 1e9,
        "latency_p50_ms": float(np.percentile(latencies, 50)) if accepted else None,
        "latency_p95_ms": float(np.percentile(latencies, 95)) if accepted else None,
        "latency_p99_ms": float(np.percentile(latencies, 99)) if accepted else None,
        "latency_max_ms": float(np.max(latencies)) if accepted else None,
        "producer_lateness_p50_ms": float(np.percentile(producer_lateness, 50)) if len(producer_lateness) else None,
        "producer_lateness_p95_ms": float(np.percentile(producer_lateness, 95)) if len(producer_lateness) else None,
        "h2d_event_p50_ms": float(np.percentile(h2d_times, 50)) if accepted else None,
        "gpu_beamform_power_p50_ms": float(np.percentile(gpu_times, 50)) if accepted else None,
        "power_map": power_map,
        "frame_records": completed,
        "drops": drops,
    }


## Warm up, build the reference, and run the sweep

The reference is one 32-frame power-Doppler ensemble with the same beamformer and geometry. A replay map is compared with it only when the trial accepted every offered frame; then each pixel is divided by the 64 replay cycles. Dropped-frame trials are not treated as complete ensembles for the image check.

The run matrix is 2 pipeline modes × 5 requested input rates × 2,048 offered frames. At low rates, the producer schedule determines trial duration; above service capacity, the queue fills and the run drains accepted work after arrivals stop.


In [ ]:
_gpu_warmup()
reference = run_ensemble(workload, geometry, batch_size=1)["power_doppler"]
print("GPU warmup complete; reference ensemble shape:", reference.shape)

trial_results = []
for _mode in ("serialized", "overlapped"):
    for _rate in INPUT_RATES_HZ:
        print(f"Running {_mode} at {_rate:,} requested frames/s ...")
        _result = run_trial(_mode, _rate)
        if _result["dropped_frames"] == 0 and _result["accepted_frames"] == replay_frames:
            _result["normalized_map"] = _result["power_map"] / CYCLES_PER_TRIAL
            _result["image_relative_rmse"] = float(
                np.linalg.norm(_result["normalized_map"] - reference)
                / max(np.linalg.norm(reference), np.finfo(np.float64).tiny)
            )
        else:
            _result["normalized_map"] = None
            _result["image_relative_rmse"] = None
        trial_results.append(_result)
        print(
            f"  accepted={_result['accepted_frames']:,}, dropped={_result['dropped_frames']:,}, "
            f"p95={_result['latency_p95_ms']:.3f} ms, "
            f"deadline misses={_result['deadline_misses']:,}"
        )


## Summarize and save

Host-observed latency includes queue wait, pinned staging, host scheduling, and GPU completion detection. H2D and beamform/power timings are CUDA event measurements, so they are not additive substitutes for end-to-end latency. A frame is counted as a deadline miss if completion occurs after its scheduled arrival plus one requested input period.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

summary_columns = [
    "mode", "requested_rate_hz", "actual_producer_rate_hz", "accepted_frames",
    "dropped_frames", "deadline_misses", "max_waiting_queue_depth",
    "latency_p50_ms", "latency_p95_ms", "latency_p99_ms", "latency_max_ms",
    "producer_lateness_p95_ms", "h2d_event_p50_ms",
    "gpu_beamform_power_p50_ms", "image_relative_rmse",
]
summary_rows = [{key: result.get(key) for key in summary_columns} for result in trial_results]
summary = pd.DataFrame(summary_rows)
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for _mode, _group in summary.groupby("mode"):
    _group = _group.sort_values("requested_rate_hz")
    _label = _mode.capitalize()
    axes[0].plot(_group["requested_rate_hz"], _group["latency_p95_ms"], marker="o", label=_label)
    axes[0].plot(_group["requested_rate_hz"], _group["latency_p99_ms"], marker="x", linestyle="--", label=f"{_label} p99")
    axes[1].plot(_group["requested_rate_hz"], _group["dropped_frames"], marker="o", label=_label)
    axes[2].plot(_group["requested_rate_hz"], _group["deadline_misses"], marker="o", label=_label)
axes[0].set(title="Host-observed frame latency", xlabel="Requested input rate (frames/s)", ylabel="Latency (ms)")
axes[0].legend(fontsize=8)
axes[1].set(title="Dropped arrivals", xlabel="Requested input rate (frames/s)", ylabel="Frames dropped")
axes[1].legend()
axes[2].set(title="Deadline misses", xlabel="Requested input rate (frames/s)", ylabel="Accepted frames missing deadline")
axes[2].legend()
for _axis in axes:
    _axis.grid(True, alpha=0.3)
fig.tight_layout()
latency_plot_path = RESULTS_DIR / "exercise5-streaming-latency.png"
fig.savefig(latency_plot_path, dpi=180, bbox_inches="tight")
plt.show()

serializable_results = []
power_maps = {}
for _index, _result in enumerate(trial_results):
    _serializable = {key: value for key, value in _result.items() if key not in {"power_map", "normalized_map"}}
    serializable_results.append(_serializable)
    if _result["normalized_map"] is not None:
        _key = f"{_result['mode']}_{_result['requested_rate_hz']}Hz"
        power_maps[_key] = _result["normalized_map"].astype(np.float32)

json_path = RESULTS_DIR / "exercise5-streaming-summary.json"
json_path.write_text(json.dumps({
    "runtime": runtime_metadata,
    "dataset": workload.data_provenance,
    "configuration": {
        "cycles_per_trial": CYCLES_PER_TRIAL,
        "recorded_frame_count": frame_count,
        "replayed_frames_per_trial": replay_frames,
        "nominal_recorded_prf_hz": REPLAYED_PRF_HZ,
        "requested_rates_hz": list(INPUT_RATES_HZ),
        "queue_capacity": QUEUE_CAPACITY,
        "drop_policy": "drop newest when the waiting queue is full",
        "deadline": "scheduled arrival plus one requested input period",
        "nvtx_available": NVTX_AVAILABLE,
    },
    "results": serializable_results,
}, indent=2))
npz_path = RESULTS_DIR / "exercise5-streaming-power-maps.npz"
np.savez_compressed(npz_path, **power_maps)
print("Saved:", json_path)
print("Saved:", npz_path)
print("Saved:", latency_plot_path)


## Nsight Systems check and interpretation

This notebook adds one NVTX range per mode/rate trial when CuPy's NVTX wrapper is available. Capture a short run from the connected Colab terminal after checking the installed `nsys` version and the current notebook-kernel process. The correct attach or launch method depends on the runtime's Nsight Systems installation, so verify that against the T4 session before running the trace. Save the `.nsys-rep` under `artifacts/` and use the timeline to confirm whether the H2D copies overlap the beamforming kernels. The timing table alone cannot establish overlap.

When interpreting the result, compare both modes at the same requested rate and check actual producer rate and lateness first. If the source thread cannot sustain the requested schedule, its jitter limits what the GPU comparison can show. A bounded queue may keep memory stable while dropping frames; queue capacity, drops, and the deadline definition must stay visible beside latency percentiles. Do not call the replayed 10 kHz source a real-time scanner capture.

**Interpretations and takeaways:** fill this section after the T4 run and Nsight timeline review, using the measured crossover rate, tail latency, producer jitter, drop counts, image check, and verified overlap evidence.
